# Blackhand on Google Colab

Runs the Blackhand gateway inside this Colab runtime and exposes it through
**your own Cloudflare Tunnel** — a named tunnel, so the hostname stays the
same every time the runtime restarts.

**What you need**
- A Cloudflare named tunnel: a `config.yml` and a credentials JSON.
- A client API key for the gateway (this notebook makes one up if you do not
  bring your own).

**What to do**
1. Fill in the three cells below: the secrets, your `config.yml`, your
   credentials JSON.
2. Run every cell from top to bottom.
3. The last cells print the endpoint to use.

The runtime's data directory is ephemeral. Everything durable comes from the
environment, so a fresh runtime reproduces itself: the API key, the settings
and the tunnel are all restored from the cells below.

## 1. Secrets

`INITIAL_PASSWORD` is the dashboard password. `ROUTER_API_KEY` is the key your
clients use to call the gateway — anything you send to the endpoint has to
carry it.

In [ ]:
import json
import os
import secrets as _secrets

# The dashboard password. Change it to something you remember.
INITIAL_PASSWORD = "blackhand"

# The client API key. Leave it empty to have one generated for this run.
ROUTER_API_KEY = ""

# A signing secret for the dashboard session. Generated per run is fine on
# Colab: the session cookie does not have to survive a restart.
JWT_SECRET = _secrets.token_hex(32)

os.environ["INITIAL_PASSWORD"] = INITIAL_PASSWORD
os.environ["JWT_SECRET"] = JWT_SECRET
os.environ["ROUTER_API_KEY"] = ROUTER_API_KEY or ("sk-" + _secrets.token_hex(24))

# The settings the gateway starts with. The keys are the ones the dashboard
# writes, so a value set here behaves exactly as if you had set it there.
os.environ["ROUTER_SETTINGS_JSON"] = json.dumps(
    {
        "requireLogin": True,        # keep the dashboard behind the password
        "requireApiKey": True,       # clients must send ROUTER_API_KEY
        "rtkEnabled": False,         # no prompt compression
        "cavemanEnabled": False,
        "ponytailEnabled": False,
        "tunnelDashboardAccess": True,  # the tunnel is the only way in here
    }
)

print("INITIAL_PASSWORD =", INITIAL_PASSWORD)
print("ROUTER_API_KEY   =", os.environ["ROUTER_API_KEY"])

## 2. Your tunnel `config.yml`

Replace the placeholder below with the contents of your `config.yml`.

The `ingress` entry's `hostname` is the address your clients will call, and
the `service` has to point at the gateway inside this runtime — `http://127.0.0.1:20130`.
The `tunnel` and `credentials-file` lines should name your tunnel id and the
credentials file; the bootstrap writes the credentials next to this file and
points the config at it.

In [ ]:
TUNNEL_CONFIG_YML = """
tunnel: your-tunnel-id
credentials-file: /content/credentials.json
ingress:
  - hostname: blackhand.your-domain.example
    service: http://127.0.0.1:20130
"""

## 3. Your tunnel credentials JSON

Replace the placeholder below with the contents of your credentials file — the
one Cloudflare gives you when you create the tunnel
(`{"AccountTag": ..., "TunnelSecret": ..., "TunnelID": ...}`).

Keep this cell private. It holds the secret that lets the gateway run your
tunnel, and a Colab notebook can be shared or exported.

In [ ]:
TUNNEL_CREDENTIALS_JSON = """
{
  "AccountTag": "00000000000000000000000000000000",
  "TunnelSecret": "REPLACE_WITH_YOUR_TUNNEL_SECRET",
  "TunnelID": "REPLACE_WITH_YOUR_TUNNEL_ID"
}
"""

## 4. Write the tunnel files and point the gateway at them

In [ ]:
from pathlib import Path

content = Path("/content")
content.mkdir(parents=True, exist_ok=True)

config_path = content / "config.yml"
credentials_path = content / "credentials.json"

config_path.write_text(TUNNEL_CONFIG_YML.strip() + "\n")
credentials_path.write_text(TUNNEL_CREDENTIALS_JSON.strip() + "\n")

os.environ["CF_TUNNEL_MODE"] = "named"
os.environ["CF_TUNNEL_CONFIG_FILE"] = str(config_path)
os.environ["CF_TUNNEL_CREDENTIALS_FILE"] = str(credentials_path)
os.environ["DATA_DIR"] = str(content / "blackhand-data")
os.environ["PORT"] = "20130"

print(config_path.read_text())
print("credentials written:", credentials_path.stat().st_size, "bytes")

## 5. Fetch the gateway

The bootstrap resolves a runnable binary: one you placed in `/content`, a
cached build, a prebuilt `linux/amd64` release, or — as a fallback — a source
build that installs Go and Bun first. It also installs `cloudflared`, which
Colab does not ship.

If the clone below fails, the fork has not been published yet; see the note
under the cell.

In [ ]:
import shutil
import subprocess

REPO_URL = os.environ.get("BLACKHAND_REPO_URL", "https://github.com/Yoganataa/9router-go.git")
REPO_DIR = content / "9router-go"

# Tell the bootstrap where the source lives, so it finds go.mod regardless of
# the directory it is started from.
os.environ["BLACKHAND_REPO"] = str(REPO_DIR)

if not REPO_DIR.exists():
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("clone failed:", result.stderr.strip()[-500:])
        print(
            "\nThe fork is not published yet. Either push it (git push origin blackhand),\n"
            "or point BLACKHAND_REPO_URL at wherever it lives, and re-run this cell."
        )
    else:
        print("cloned into", REPO_DIR)
else:
    print("already cloned:", REPO_DIR)

In [ ]:
# Run the bootstrap with this process environment. A notebook magic ("!" or
# "%") would start a fresh shell, and the environment variables set above
# would not travel with it.
result = subprocess.run(
    [sys.executable, "scripts/colab-bootstrap.py"],
    cwd=str(REPO_DIR),
    env=dict(os.environ),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit(f"the bootstrap failed with exit code {result.returncode}")

## 6. What to use

The endpoint is your tunnel hostname, and every request carries the API key
from cell 1.

Run the cell below for a copy-and-paste summary, then a live check that the
gateway answers through the tunnel.

In [ ]:
import json as _json
from pathlib import Path

status = _json.loads(Path(os.environ["DATA_DIR"], "status.json").read_text())
hostname = status.get("tunnelUrl")

if not hostname:
    print("No tunnel URL in the status file. Check the bootstrap output above.")
else:
    endpoint = hostname.rstrip("/") + "/v1"
    print("Endpoint :", endpoint)
    print("API key  :", os.environ["ROUTER_API_KEY"])
    print("Password :", INITIAL_PASSWORD)
    print()
    print("Dashboard:", hostname)
    print()
    print("curl example:")
    print(f'  curl {endpoint}/chat/completions \\')
    print(f'    -H "Authorization: Bearer {os.environ["ROUTER_API_KEY"]}" \\')
    print('    -H "Content-Type: application/json" \\')
    print('    -d \'{"model": "oc/muse-spark-1.3-contributor-free", "messages": [{"role": "user", "content": "hi"}]}\'')

In [ ]:
# A live check through the tunnel: list the models the gateway can serve.
import json as _json
import urllib.request

endpoint = hostname.rstrip("/") + "/v1"
request = urllib.request.Request(
    endpoint + "/models",
    headers={"Authorization": "Bearer " + os.environ["ROUTER_API_KEY"]},
)
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        models = _json.loads(response.read())
    print("the gateway answered through the tunnel with", len(models.get("data", [])), "models")
except Exception as error:
    print("the check failed:", error)